# 🚀 10주차 실습 (2026-06-11)

오늘의 학습 주제에 맞춰 실습을 진행할 수 있도록 준비된 노트북입니다.

In [12]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

import pandas as pd
import numpy as np
from pydantic import BaseModel, Field
from config import GOOGLE_AI_API_KEY

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableSequence, RunnableParallel

CHAT_MODEL = 'google_genai:gemma-4-31b-it'
# CHAT_MODEL = 'gemini-3.1-flash-lite'
print("실습 환경 준비 완료!")

실습 환경 준비 완료!


In [13]:
model = init_chat_model(CHAT_MODEL, api_key = GOOGLE_AI_API_KEY)
model

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemma 4 31B IT', 'release_date': '2026-04-02', 'last_updated': '2026-04-02', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemma-4-31b-it', client=<google.genai.client.Client object at 0x72872ce39e50>, default_metadata=(), model_kwargs={})

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
("system", "너는 생성형 AI와 Langchain에 대해 쉽게 설명을 해주는 멘토다. 한국어로 대답해줘."),
("human", "주제:{topic}\n 대상: {audience} \n 요청: 쉬운 설명, 예시 2개, 확인 질문 1개를 작성해줘.")
])

# 1. 프롬프트 템플릿에 값을 넣어 실제 메시지 작성
messages = chat_prompt.invoke({
    "topic": "Output Parser",
    'audience': "파이썬 기초를 아는 학생"
})

# 2. 모델 호출
response = model.invoke(messages)

# 3. 모델 응답에서 문자열만 추출
parser = StrOutputParser()
answer = parser.invoke(response)

print(answer)

반가워요! 생성형 AI와 LangChain의 세계에 오신 것을 환영합니다. 파이썬 기초를 알고 계시다니, Output Parser(출력 파서)의 개념을 아주 빠르게 이해하실 수 있을 거예요.

자, 그럼 친절하게 설명해 드릴게요!

---

### 💡 Output Parser란 무엇일까요?

쉽게 말해, **"AI가 내뱉은 '말(텍스트)'을 컴퓨터가 이해하기 좋은 '데이터 형식'으로 바꿔주는 번역기"**라고 생각하면 됩니다.

우리가 ChatGPT 같은 LLM(거대언어모델)에게 질문을 하면, AI는 항상 **문자열(String)** 형태로 대답합니다. 하지만 우리가 프로그램을 만들 때는 단순한 문장보다는 **리스트(List)**나 **딕셔너리(Dictionary/JSON)** 같은 구조화된 데이터가 필요할 때가 많죠.

**예를 들어볼까요?**
- **AI의 답변:** "제가 추천하는 과일은 사과, 바나나, 포도입니다." $\rightarrow$ (이건 그냥 긴 문장이에요.)
- **우리가 원하는 데이터:** `['사과', '바나나', '포도']` $\rightarrow$ (이렇게 되어 있어야 파이썬의 `for`문으로 하나씩 꺼내 쓸 수 있겠죠?)

이처럼 **[AI의 텍스트 답변] $\rightarrow$ [파이썬 데이터 구조]**로 변환해주는 도구가 바로 **Output Parser**입니다.

---

### 🛠️ 예시로 알아보기

LangChain에서 자주 쓰이는 두 가지 사례를 들어볼게요.

#### 예시 1: 쉼표로 구분된 리스트로 받기 (`CommaSeparatedListOutputParser`)
AI에게 여러 개의 아이템을 추천해달라고 하고, 이를 파이썬 **리스트**로 바로 받고 싶을 때 사용합니다.

*   **질문:** "한국의 유명한 도시 3곳을 쉼표로 구분해서 알려줘."
*   **AI의 원래 답변:** `"서울, 부산, 제주"` (단순 문자열)
*   **Output Parser 적용 후:** `['서울', '부산', '제주']` (파

In [ ]:
class CustomerIssue(BaseModel):
    sentiment: Literal["긍정", "중립", "부정"] = Field(
        description="고객 후기의 전체 감성")
    category: Literal["배송", "결제", "상품", "CS", "기타"] =Field(
        description="가장 중요한 이슈 카테고리")
    priority: int = Field(
        description="1은 낮음, 5는 매우 긴급",
        ge=1,
        le=5,
    )
    summary: str = Field(description="고객 이슈 요약")
    next_action: str = Field(description="담당자가 취해야 할 다음 조치")

# with_structured_output : LLM 출력이 Pydantic 모델 형태로 반환
issue_model = model.with_structured_output(CustomerIssue, method="json_schema")

review = "배송은 빨랐지만 박스가 찢어져 있고, 문의를 했지만 고객센터 답변이 3일째 없습니다."

issue = issue_model.invoke(review)

issue

CustomerIssue(sentiment='부정', category='CS', priority=4, summary='배송 박스 파손 및 고객센터 응답 지연', next_action='파손 상품 확인 및 지연된 문의에 대해 즉시 답변 및 보상 처리')

## 🎮 실습: Pydantic을 활용한 게임 캐릭터 카드 구조화 생성

앞서 살펴본 `CustomerIssue` 분석 모델과 동일한 방식으로, 게임 캐릭터의 카드 정보를 구조화된 형태로 받아올 수 있습니다. Pydantic의 `BaseModel`을 상속받아 원하는 캐릭터 카드 필드를 정의한 후 `with_structured_output`을 사용해 봅시다.

In [ ]:
class GameCharacter(BaseModel):
    name: str = Field(description="게임 캐릭터의 이름")
    job: str = Field(description="캐릭터의 직업 (예: 전사, 마법사, 도적, 성직자 등)")
    personality: str = Field(description="캐릭터의 성격 및 태도")
    specialty: str = Field(description="캐릭터의 대표 특기 또는 필살기")
    weakness: str = Field(description="캐릭터가 가진 치명적인 약점")

# 구조화된 출력 모델 생성
character_model = model.with_structured_output(GameCharacter, method="json_schema")

# 캐릭터 설명 예시 텍스트
description = "이그니스는 호기심이 매우 많은 전설적인 화염 마법사입니다. 전장을 뒤덮는 '메테오 스트라이크'가 주특기이지만, 물 속이나 강한 비가 내리는 환경에서는 마법을 전혀 쓰지 못하는 치명적인 약점을 가지고 있습니다."

# LLM 호출 및 파싱
character_card = character_model.invoke(description)
character_card

GameCharacter(name='이그니스', job='화염 마법사', personality='호기심이 매우 많음', specialty='메테오 스트라이크', weakness='물 속이나 강한 비가 내리는 환경')

In [ ]:
# 이번에는 설명글을 제공하는 대신, 프롬프트 템플릿을 연결하여 LLM이 새로운 캐릭터를 창작하도록 구성해 보겠습니다.
character_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 판타지 게임의 캐릭터 크리에이터입니다. 사용자의 요청에 맞는 창의적이고 상세한 캐릭터 카드를 작성해 주세요."),
    ("human", "{user_request}")
])

# LCEL 체인 결합
character_chain = character_prompt | character_model

# 실행
new_character = character_chain.invoke({"user_request": "어둠의 숲을 지키는 고독한 엘프 궁수를 만들어줘."})
new_character

GameCharacter(name='실리안 나이트셰이드', job='숲의 파수꾼 엘프 궁수', personality='침묵을 사랑하며 타인에게 냉소적이지만, 숲의 생명체들에게는 한없이 다정한 면모를 지닌 고독한 수호자', specialty='그림자 화살: 어둠 속에 숨어 적의 심장을 정확히 꿰뚫는 보이지 않는 화살', weakness='숲을 벗어난 개방된 지형에서의 심리적 불안감과 낮은 방어력')

In [9]:
concept_prompt = ChatPromptTemplate.from_messages([
    ('system', '너는 수학을 쉽게 설명해주는 수학 강사이다.'),
    ('human', '개념: {concept}, 난이도: {level}, 설명, 예시, 주의점을 작성해줘.' )
])

concept_chain = concept_prompt | model | parser

result = concept_chain.invoke({
    "concept": "미분",
    "level" : "입문"
})

print(result)

안녕하세요! 수학을 쉽고 친절하게 알려주는 수학 강사입니다. 

많은 학생들이 '미분'이라는 단어를 들으면부터 겁을 먹곤 하지만, 사실 미분의 핵심 아이디어는 우리가 일상생활에서 이미 경험하고 있는 아주 단순한 개념이에요. 

**"미분 = 순간적인 변화를 포착하는 것"**

이 문장만 기억하고 따라오세요! 입문자를 위한 미분 가이드를 시작합니다.

---

### 1. 설명: 미분이란 무엇인가? 🧐

**① 변화율 (Rate of Change)**
우선 '변화율'이라는 말을 이해해야 합니다. 예를 들어, 여러분이 서울에서 부산까지 차를 타고 간다고 해봅시다. 
*   **평균 변화율:** 전체 거리(400km)를 전체 시간(4시간)으로 나누면 시속 100km죠? 이것이 '평균 변화율'입니다.

**② 순간 변화율 (Instantaneous Rate of Change)**
그런데 운전을 하다 보면 속도계의 바늘이 계속 움직이죠? 어떤 때는 80km/h였다가, 추월할 때는 120km/h가 되기도 합니다. 바로 **'특정 찰나의 순간'에 내가 얼마나 빠르게 변하고 있는가**를 나타내는 것이 바로 **미분**입니다.

**③ 기하학적 의미 (그래프에서의 미분)**
그래프에서 미분은 **'접선의 기울기'**를 의미합니다.
*   곡선 위의 한 점에 아주 짧은 자를 대고 직선을 그었을 때, 그 직선이 얼마나 가파른지를 측정하는 것이죠. 
*   기울기가 크면 급격하게 변하는 것이고, 기울기가 0이면 잠시 멈춘(평평한) 상태인 것입니다.

---

### 2. 예시: 쉽게 이해하는 미분 💡

#### [실생활 예시] 자동차 속도계
*   **위치 함수:** 내가 지금 어디에 있는지를 나타내는 식
*   **미분 결과(속도):** 위치 함수를 미분하면 '속도'가 됩니다. (위치가 시간에 따라 어떻게 변하는지를 보는 것)
*   **한 번 더 미분(가속도):** 속도를 다시 미분하면 '가속도'가 됩니다. (속도가 시간에 따라 어떻게 변하는지를 보는 것)

#### [수학적 

In [ ]:
# 1. 초등학생 대상 체인 정의
elementary_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 초등학생 눈높이에 맞춰 비유로 설명하는 교사다."),
    ("human", "{concept}에 대해 설명해줘.")
])
elementary_chain = elementary_prompt | model | StrOutputParser()
# 2. 고등학생 대상 체인 정의
highschool_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 고등학생을 위해 수식과 정의 위주로 설명하는 강사다."),
    ("human", "{concept}의 정의와 수식을 설명해줘.")
])
highschool_chain = highschool_prompt | model | StrOutputParser()
# 3. RunnableParallel을 사용하여 두 체인을 병렬 결합
# 입력값인 {"concept": "..."}가 elementary_chain과 highschool_chain에 동시에 전달됩니다.
parallel_chain = RunnableParallel(
    child_version=elementary_chain,
    student_version=highschool_chain
)
# 4. 실행 (두 번의 API 호출이 병렬로 수행됩니다)
result = parallel_chain.invoke({"concept": "미분"})
# 5. 결과 확인
print("--- 초등학생 버전 ---")
print(result["child_version"])
print("\n--- 고등학생 버전 ---")
print(result["student_version"])